# 4 Co-creating an image-analysis function with an LLM
In this module, scientist and language model work together to create a reusable function that segments the cilia, dilates the mask, and returns the protein-channel intensity sum, mask area, and mean intensity.

We use three complementary pedagogical tools:

- **Copy–modify–create** guides the progression: copy established processing steps or a draft, modify them for the agreed task, then create a reusable solution by combining those ideas.
- **Predict–code–reflect** guides each activity: anticipate the result, develop and run code, then reflect on the evidence. During planning, a written plan takes the place of code: predict, co-create the artefact, then reflect on its quality.
- **Initiate–refine** guides the collaboration with the LLM. Start with a prompt asking for an initial proposal or presenting a nearly complete draft. Review the response against the task and agreed decisions. Give specific feedback and decisions in a follow-up prompt, then inspect the revision. Repeat when needed until the team can justify accepting the result.

Develop the plan with the LLM, and accept it when you can explain why it suits your measurement. Keep written plans, prompts, responses, and revisions in this notebook so a colleague can understand how the result was obtained. Reflection concerns the existing results and decisions; further revision belongs in the activity before its Reflect prompt.

This solution notebook offers one set of choices and an author-written reference implementation. It is not a record of an LLM interaction. During class, students should record their own prompt and actual LLM response; different solutions are reasonable when their requirements and checks agree.

## A “brain-rot” warning: keep mental ownership
Working with an LLM can distance your ideas from the solution. The model can produce decisions and code faster than you can understand them, making it tempting to accept plausible answers and become a passive reviewer. “Brain-rot” refers to this risk of outsourcing the thinking as well as the typing.

Responsibility for the quality of the solution remains entirely yours as the scientist carrying out the analysis. The LLM's confidence, a convincing explanation, and code that runs do not establish that the result answers your question. You need conscious steps to become the mental owner of the result:

- Form your own prediction and identify the important requirements before accepting a proposal.
- Review the model's choices against your intended measurement, and ask about assumptions you cannot justify.
- Trace each processing step and account for it in your own words, including what the inputs, parameters, and outputs mean.
- Check expected results independently and examine failures rather than relying on the model's claim that its code is correct.
- Judge whether you could adapt the solution to a changed requirement or diagnose a wrong result. If not, identify the part you still need to understand.

The purpose of co-creation is to use the LLM's contributions while retaining your own judgement. Initiate–refine and predict–code–reflect are deliberate ways to stay engaged and take ownership of the accepted solution.

## 4.1 Base case: protein intensity near cilia
The base case is a direct calculation of protein-channel intensity near cilia. `Image_34362.tif` shows the cilia mainly in green, and `Image_34363.tif` shows the protein of interest mainly in red. The images represent corresponding pixel positions; one image's mask can therefore select pixels in the other.

An RGB image is a table indexed by row, column, and colour channel. Channels 0, 1, and 2 are red, green, and blue. The expression `cilia[:,:,1]` selects all rows and columns of the green channel, giving a two-dimensional array. `protein[:,:,0]` selects the red channel.

Open this notebook with `4_LLM` as the working directory. The code reads the two local files, selects the channels, and displays the images. NumPy handles arrays, scikit-image provides image reading and morphology, and Matplotlib displays figures.

**Predict:** Before displaying the images, where do you expect to see bright cilia and bright protein signals: in the red or green channel? Why must the two images show the same field of view at corresponding pixel positions?

In [ ]:
from skimage import io, morphology
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams['image.cmap'] = 'gray'

# Read the two images included alongside this notebook.
cilia = io.imread('Image_34362.tif')
protein = io.imread('Image_34363.tif')
green = cilia[:,:,1]
red = protein[:,:,0]
plt.imshow(cilia)
plt.title('Cilia')
plt.show()
plt.imshow(protein)
plt.title('Protein')
plt.show()


**Reflect:** Do the displayed colours agree with the intended cilia and protein channels? Why do the images need both matching dimensions and corresponding positions for this measurement?

The direct calculation has three steps:

1. **Segment:** `green > chosenThreshold` creates a Boolean mask. A pixel is True only when its green intensity is strictly greater than the threshold.
2. **Dilate:** expand the selected pixels with a disk of the chosen radius, measured in pixels. The expanded mask includes the original cilia pixels and their neighbourhood; expansion stops at the image boundary.
3. **Measure:** multiply red intensities by the expanded mask, retaining selected values and setting the others to zero. Their sum is the total intensity. The number of True pixels is the mask area. Dividing the sum by the area gives mean intensity inside the mask. If no pixels are selected, the mean is undefined and recorded as `np.nan`.

Threshold 75 and radius 20 are illustrative choices, not automatically optimal. This base-case calculation supplies comparison values for the eventual function. The function will accept those choices as arguments so it can be reused without editing its body.

**Predict:** If you increase the radius around the cilia, how do you expect the selected area and total protein intensity to change? Will the mean intensity necessarily increase too?

In [ ]:
# Direct base-case calculation with explicit threshold and radius choices.
chosenThreshold = 75
chosenRadius = 20
segm = green > chosenThreshold
msk = morphology.dilation(segm, morphology.disk(chosenRadius))
redMsk = red.astype(float) * msk
referenceTotal = redMsk.sum()
referenceArea = msk.sum()
if referenceArea == 0:
    referenceMean = np.nan
else:
    referenceMean = referenceTotal / referenceArea
print('Reference intensity sum:', referenceTotal)
print('Reference mask area (pixels):', referenceArea)
print('Reference mean intensity:', referenceMean)
plt.imshow(redMsk)
plt.title('Protein intensities inside the expanded cilia mask')
plt.show()


**Reflect:** What does the sum measure, what does the area count, and how is the mean obtained? Which aspects depend on the threshold and radius, and why is this fluorescence measurement not automatically an absolute amount of protein?

## 4.2 Plan the function: requirements, design, and specification
A plan connects the scientific question to the calculation: requirement analysis describes what you want to measure, design choices explain how to measure it, and a code specification states the inputs, processing steps, and expected results. For a measurement of protein intensity near cilia, this includes which pixels are selected, the radius and its units, the meaning of the returned measurements, and what happens when no pixels are selected. Choices such as keeping file loading and plotting outside the calculation support reuse.

### Co-create, review, and refine
To work effectively with an agent or an LLM, explain what you want to measure and develop the solution through feedback. Start by asking for a proposal or sharing your own draft for review. Include the relevant image-analysis steps and decisions you have already made, since the model may not have access to this notebook.

For example, an initial prompt could be:

```text
We are co-creating a reusable function that thresholds a cilia image, dilates its mask, and returns protein-channel intensity sum, mask area, and mean. Help us write a short plan for this measurement. Here are our intended measurement and settings: [include the relevant context]. Propose an initial version and identify unresolved choices. Keep the plan concise; do not write implementation code yet.
```

You can also begin with your own draft plan and ask the LLM to suggest improvements. In either case, read the response and consider whether the proposed calculation matches the measurement you want to make. Identify anything unclear or missing, or a choice you would like explained, and give specific feedback. A follow-up prompt could be:

```text
Our review found [specific issue or decision]. We intend [behaviour or choice] because [reason or independently checked evidence]. Revise the plan accordingly and explain how this affects the measurement.
```

Read the revision and continue until you can explain why the plan suits your measurement. Save the agreed plan and a brief record of your feedback in the space below. Use the same approach to discuss the quality conditions and code in the following assignments, asking for the output needed at each step.

**Assignment 4a:** Co-create one concise plan for the reusable base-case measurement. Include the requirements, reasons for the main design choices, and enough detail for a colleague to understand how the measurement will be calculated. Use a short paragraph or bullet list.

**Predict:** You want a colleague to repeat your measurement. What would you tell them about which pixels to include and what the three results mean? If no cilia pixels are selected, should the mean intensity be zero or undefined, and why?

**Example agreed plan**

- **Measurement:** Pool the protein intensities within the cilia and their expanded surroundings. Supply the corresponding green and red channel images, a threshold, and a radius in pixels to a reusable function.
- **Selected region:** Include green pixels strictly above the threshold, then expand with a disk of the chosen radius. Expansion stops at the image edges; radius 0 leaves the selection unchanged.
- **Results:** Return intensity sum, area in pixels, and mean intensity, in that order. If no pixels are selected, return zero sum, zero area, and an undefined mean (`np.nan`). Convert red values to floating-point numbers before arithmetic to handle large intensity values reliably.
- **Reuse:** Use two nonempty, two-dimensional images of the same size with finite, nonnegative intensities, a finite threshold, and a whole-number radius of zero or greater. The images must represent corresponding positions. Leave the images unchanged and keep loading and plotting outside the function. Report an error for different image sizes, non-2D images, or an invalid radius. Document the assumptions in the function. Choosing a biological threshold, subtracting background, and measuring individual cilia are outside this calculation.

**Reflect:** Does your plan clearly explain which pixels are included and what happens when none are selected? What changed after discussing it with the LLM, and why did you accept the revised plan?

## 4.3 Quality conditions before code generation
Quality conditions describe what would give you confidence in the measurement. They follow from your scientific question and agreed plan. They can include whether the selected region matches the cilia and their surroundings, whether the results agree with the direct calculation, and whether changing the radius has an understandable effect.

An expectation based on the images helps you interpret a result, but agreement with that expectation alone does not establish correctness. Confidence also comes from inspecting the selected region, comparing calculations, and recording assumptions and limitations so a colleague can assess the work.

**Assignment 4b:** Co-create a short list of quality conditions for the measurement. Include what you expect from these images, how you will compare the results with the direct calculation, and what you need to inspect before interpreting them.

**Predict:** Look at the cilia and protein images. Do you expect the mean protein intensity near the cilia to be higher or lower than in the image as a whole? What features in the images lead you to that expectation, and what measured result would surprise you?

**Example quality conditions**

| Condition | Evidence to examine |
|---|---|
| Selected region makes sense | Inspect the thresholded and expanded masks alongside the cilia image; the expanded region should include the cilia and their chosen neighbourhood |
| Same calculation gives the same result | With the same images, threshold, and radius, the function should match the three measurements in section 4.1 |
| Result can be interpreted from the images | Compare the mean near the cilia with the mean across the protein image; investigate a surprising difference rather than assuming it is an error |
| Radius has an understandable effect | Increasing the radius should not reduce selected area or total intensity for these nonnegative images; mean intensity may rise or fall depending on the added pixels |
| No selected pixels are handled clearly | A threshold at or above the maximum green intensity should give zero sum, zero area, and an undefined mean |
| A colleague can repeat the calculation | Save the images used, channel choices, threshold, radius, code, and example results; explain the processing steps and assumptions |

Agreement with the direct calculation checks that the function reproduces it. It does not establish that the threshold identifies every cilium or that fluorescence measures absolute protein concentration.

**Reflect:** How will your quality conditions help you decide whether the result is credible? Which can you assess from the images and measured values, and which need further evidence? If a result differs from your expectation, what would you investigate before accepting or rejecting it?

## 4.4 Implement and verify
The LLM proposes code; you decide whether it carries out your agreed measurement. Compare its results with the direct calculation and your expectations from the images. If a result differs, discuss the expected and observed values with the LLM and investigate the cause before changing the code or the plan.

Copy only the Python code into the code cell, leaving out surrounding text and the triple backticks used to mark code in a response. Save a brief record of any revisions. Restart the notebook's kernel and run all cells in order to check that the analysis works from a fresh start.

**Assignment 4c:** Ask the LLM to write code for your agreed plan from Assignment 4a and to include calls that help you assess the quality conditions from Assignment 4b. Paste and run the code below. Compare the three measurements with section 4.1, and compare the mean near the cilia with the mean across the whole protein image. Call the function with a different radius and examine how the results change. Ask the LLM to explain any processing steps you do not understand, and refine the code where needed.

**Predict:** When you run the new function on the same images with threshold 75 and radius 20, what results do you expect compared with section 4.1? What result would make you question whether it is measuring the same thing?

In [ ]:
# Author-written reference implementation, not a recorded response from an LLM.
def measure_near_cilia(green_image, red_image, threshold, radius):
    """Return intensity sum, mask area (pixels), and mean near the cilia.

    Images must be same-shaped, nonempty 2D NumPy arrays with finite,
    nonnegative values; threshold must be a finite number. Segment with
    green_image > threshold and expand with a disk of radius pixels.
    Inputs are unchanged. An empty mask returns (0, 0, np.nan).
    """
    if green_image.ndim != 2 or red_image.ndim != 2:
        raise ValueError('Images must be two-dimensional')
    if green_image.shape != red_image.shape:
        raise ValueError('Images must have equal shapes')
    if not isinstance(radius, (int, np.integer)) or isinstance(radius, (bool, np.bool_)) or radius < 0:
        raise ValueError('Radius must be a nonnegative integer')

    segmentation = green_image > threshold
    mask = morphology.dilation(segmentation, morphology.disk(radius))
    masked_red = red_image.astype(float) * mask
    total = masked_red.sum()
    area = mask.sum()
    if area == 0:
        mean = np.nan
    else:
        mean = total / area
    return total, area, mean

# Use the same arrays and parameter choices as the direct base-case calculation.
total, area, mean = measure_near_cilia(green, red, chosenThreshold, chosenRadius)
print('Base-case sum, area, mean:', referenceTotal, referenceArea, referenceMean)
print('Function intensity sum:', total)
print('Function mask area (pixels):', area)
print('Function mean intensity:', mean)
# Compare the printed function results with the base-case values above.

# Compare with the whole protein image, as predicted in section 4.3.
print('Mean protein intensity across the whole image:', red.mean())
print('Mean protein intensity near the cilia:', mean)

# Inspect the selected regions alongside the cilia image.
plt.figure(figsize=(12,4))
for panel, image, title in [(1, green, 'Cilia channel'),
                            (2, segm, 'Thresholded cilia'),
                            (3, msk, 'Expanded measurement region')]:
    plt.subplot(1,3,panel)
    plt.imshow(image)
    plt.title(title)
plt.tight_layout()
plt.show()

# Select no pixels: expected sum 0, area 0, and undefined mean.
print('No selected pixels; expected (0, 0, nan):',
      measure_near_cilia(green, red, green.max(), chosenRadius))

# Reuse the same function with a different radius, without editing its body.
largerRadius = chosenRadius + 5
largerTotal, largerArea, largerMean = measure_near_cilia(green, red, chosenThreshold, largerRadius)
print('Original radius:', chosenRadius, 'Larger radius:', largerRadius)
print('Intensity sums:', total, largerTotal)
print('Mask areas:', area, largerArea)
print('Mean intensities:', mean, largerMean)

**Reflect:** Do the results agree with the direct calculation and with what you expected from the images? How did changing the radius affect your interpretation? Which quality conditions from 4b are supported, and what remains uncertain? Can you explain how the code selects the pixels and calculates the measurements?

**Reference provenance:** This is an author-written example, not a recorded LLM response.

## 4.5 Evaluate the co-created result
Quality concerns the whole process: the agreed plan, quality conditions, collaboration, and implementation. A working program alone does not establish that the intended measurement is correct.

Reuse means applying the function in another supported situation; maintenance means understanding and changing it without accidentally altering its behaviour. A colleague may use the same calculation for another image pair, while measuring each cilium separately would require changes to the plan and code. Saved settings, example results, and explanations help them make those changes and assess their effects.

**Assignment 4d:** Give a brief quality review of the agreed plan, quality conditions, and implementation. Include one strength, one limitation, and what a colleague would need to reuse or maintain the result.

**Predict:** Imagine a colleague repeats your analysis and obtains a different mean intensity. Which saved information would help you work out why: the images and channels used, the threshold and radius, or the steps in the calculation? Explain your choices.

**Example quality review**

| Part of the work | Strength | Limitation or future need |
|---|---|---|
| Agreed plan | Defines the selected region, settings, units, and meaning of the results | Does not justify the biological threshold or calibrate fluorescence |
| Evidence | Displays the selected region and prints function and direct-calculation results for comparison | Matching calculations can share the same scientific assumptions; visual inspection remains necessary |
| Interpretation | Compares the local and whole-image means and explores a different radius | Background intensity and the chosen neighbourhood affect the conclusion |
| Code and handover | Keeps a reusable function, image choices, settings, and example calls together | A colleague should rerun it and inspect the results before changing the analysis |

For further development, supply this notebook and its images, record the software versions used, and explain known limitations. A colleague can build confidence by reproducing the example results, inspecting the masks, and comparing results before and after a change.

**Reflect:** Where was your judgement essential, and what did the LLM contribute? Do you understand the accepted result well enough to take responsibility for its quality? If you gave this code to a colleague for further development, what would you include so they could understand and change it? How could they check the results themselves and judge whether to trust the measurement? Consider the example images, parameter choices, expected results, and known limitations.